### GPU Verification
Run this cell after setting your hardware accelerator to GPU to ensure it is connected.

In [1]:
import torch

if torch.cuda.is_available():
    print(f" GPU connected: {torch.cuda.get_device_name(0)}")
else:
    print(" No GPU connected. Please go to Runtime > Change runtime type and select a GPU.")

 GPU connected: NVIDIA A100-SXM4-80GB


# Preparing Text Data


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import os
import sys

module_path = '/content/drive/MyDrive/LM'
if module_path not in sys.path:
    sys.path.append(module_path)

from datasets import load_dataset, Dataset, load_from_disk
from BPETokenizer import BPETokenizer as tk
from SlidingWindowDataset import create_data_loader
from TokenAndPositionEmbedding import TokenAndPositionEmbedding
import itertools
import torch

from tqdm import tqdm
import logging

In [4]:
logging.basicConfig(
    level= logging.INFO,
    format= '%(asctime)s - %(levelname)s - %(message)s',
    datefmt= '%Y-%m-%d %H:%M:%S',
    force= True
)

In [5]:
SLICE_SIZE = 1_000_000
VOCAB_TRAIN_SIZE = 50_000
NUM_MERGES = 1000

CONTEXT_SIZE = 1024
STRIDE = 1024
EMBED_DIM = 768
BATCH_SIZE = 16

REGEX_PATTERN = r"""'(?:[sdmt]|ll|ve|re)| ?[a-zA-Z]+| ?[0-9]+| ?[^\sA-Za-z0-9]+|\s+(?!\S)|\s+"""

### Data Ingestion & Caching

In [7]:

gdrive_save_path = "/content/drive/MyDrive/LM/local_fineweb_data/hf_format"

if not os.path.exists(gdrive_save_path):
    print(f"Streaming {SLICE_SIZE:,} documents from FineWeb-Edu...")
    dataset_dict = load_dataset("HuggingFaceFW/fineweb-edu", name="sample-10BT", streaming=True)

    # Use a generator to avoid loading everything into RAM at once
    def data_generator():
        for example in dataset_dict['train'].take(SLICE_SIZE):
            yield example

    print("Converting dataset using memory-efficient generator...")
    local_dataset = Dataset.from_generator(data_generator)

    print(f"Saving dataset to Google Drive: {gdrive_save_path}")
    os.makedirs(os.path.dirname(gdrive_save_path), exist_ok=True)
    local_dataset.save_to_disk(gdrive_save_path)
else:
    print("Loading existing dataset from Google Drive...")

local_dataset = load_from_disk(gdrive_save_path)
print(f"Dataset ready. Total documents: {len(local_dataset):,}")

Loading existing dataset from Google Drive...
Dataset ready. Total documents: 1,000,000


### Parallel Tokenization

In [8]:
TOKENIZER_PATH = "bpe_tokenizer.json"

logging.info("Checking for existing tokenizer")
tokenizer = tk.load(TOKENIZER_PATH)

if tokenizer is None:
  logging.info(f"Training BPE Tokenizer on a subset of {VOCAB_TRAIN_SIZE:,} documents")

  vocab_sub_sample =  local_dataset.select(range(VOCAB_TRAIN_SIZE))
  tokenizer = tk(REGEX_PATTERN, num_merges=NUM_MERGES)

  tokenizer.build_vocab(vocab_sub_sample)
  tokenizer.save(TOKENIZER_PATH)

  logging.info("Tokenizer training complete and saved to disk.")

else:
  logging.info("Pre-trained tokenizer loaded successfully.")

logging.info(f"Total vocabulary size: {tokenizer.vocab_size:,}")

logging.info(f"Encoding dataset in parallel using {os.cpu_count()} cores")
tokenized_dataset = local_dataset.map(tokenizer.encode_row, num_proc=os.cpu_count())

logging.info("Flattening token IDs into a single master sequence")
all_token_ids = list(itertools.chain.from_iterable(tokenized_dataset['token_ids']))
logging.info(f"flattening completed. Sequence length: {len(all_token_ids):,} tokens.")

2026-05-29 17:04:32 - INFO - Checking for existing tokenizer
2026-05-29 17:04:32 - INFO - Training BPE Tokenizer on a subset of 50,000 documents


Error: The file bpe_tokenizer.json was not found. Please check the path.


2026-05-29 17:29:37 - INFO - Tokenizer training complete and saved to disk.
2026-05-29 17:29:37 - INFO - Total vocabulary size: 5,163
2026-05-29 17:29:37 - INFO - Encoding dataset in parallel using 12 cores


Successfully savedd tokenizer state to bpe_tokenizer.json


2026-05-29 17:36:25 - INFO - Flattening token IDs into a single master sequence
2026-05-29 17:50:28 - INFO - flattening completed. Sequence length: 1,787,112,294 tokens.


### Sanity Check - Autogressive

In [10]:
test_context_size = 4
enc_sample = all_token_ids[100:]
x = enc_sample[:test_context_size]
y = enc_sample[1:test_context_size + 1]

print("Visualizing Autoregressive Targets:")
for index in range(1, test_context_size + 1):
    context = enc_sample[:index]
    desired = enc_sample[index]
    print(f"{tokenizer.decode(context)}  --------->  {tokenizer.decode([desired])}")

Visualizing Autoregressive Targets:
 ind  --------->  ep
 indep  --------->  end
 independ  --------->  ence
 independence  --------->   s


### PyTorch DataLoader Initialization

In [11]:
print(f"Creating DataLoader with Context Size: {CONTEXT_SIZE} and Stride: {STRIDE}...")
dataloader = create_data_loader(
    all_token_ids,
    batch_size=BATCH_SIZE,
    context_size=CONTEXT_SIZE,
    stride=STRIDE,
    shuffle=True,
    drop_last=True
)
print(f"DataLoader created successfully with {len(dataloader):,} batches.")

Creating DataLoader with Context Size: 1024 and Stride: 1024...
DataLoader created successfully with 109,076 batches.


### Sanity Check - Batch Alignment

In [12]:
x_batch, y_batch = next(iter(dataloader))

print("Input Batch Shape (X):", x_batch.shape)
print("Target Batch Shape (Y):", y_batch.shape)

# Verify that Y is shifted exactly 1 position to the right of X
print("\nFirst row, first 5 tokens of X:", x_batch[0, :5].tolist())
print("First row, first 5 tokens of Y:", y_batch[0, :5].tolist())

# Mathematically assert the shift for the entire sequence (up to the last token of X)
assert torch.all(x_batch[:, 1:] == y_batch[:, :-1]), "ERROR: Y is not properly shifted!"
print("\n✅ Assertion Passed: Y is correctly shifted by 1 token.")

Input Batch Shape (X): torch.Size([16, 1024])
Target Batch Shape (Y): torch.Size([16, 1024])

First row, first 5 tokens of X: [4693, 4306, 4350, 4189, 4195]
First row, first 5 tokens of Y: [4306, 4350, 4189, 4195, 5054]

✅ Assertion Passed: Y is correctly shifted by 1 token.


### Positional Embeddings

In [13]:
# Initialize the model's first structural layer
embedding_layer = TokenAndPositionEmbedding(
    vocab_size=tokenizer.vocab_size,
    embed_dim=EMBED_DIM,
    context_size=CONTEXT_SIZE
)


### Sanity Check - Embedding Shapes

In [14]:

# Pass our test batch through the embedding layer
embedded_batch = embedding_layer(x_batch)

print("--- FINAL PIPELINE VERIFICATION ---")
print(f"Pre-Embedding Shape (X):   {x_batch.shape}")          # [BATCH_SIZE, CONTEXT_SIZE]
print(f"Post-Embedding Shape:      {embedded_batch.shape}")   # [BATCH_SIZE, CONTEXT_SIZE, EMBED_DIM]

# Verify dimensions match our GPT-2 specs
assert embedded_batch.shape == (BATCH_SIZE, CONTEXT_SIZE, EMBED_DIM), "ERROR: Dimensionality mismatch!"
print(f"✅ Target Dimensions Achieved: Batch={BATCH_SIZE}, Sequence={CONTEXT_SIZE}, Embed_Dim={EMBED_DIM}")
print("\nPipeline status: READY FOR MULTI-HEAD ATTENTION")

--- FINAL PIPELINE VERIFICATION ---
Pre-Embedding Shape (X):   torch.Size([16, 1024])
Post-Embedding Shape:      torch.Size([16, 1024, 768])
✅ Target Dimensions Achieved: Batch=16, Sequence=1024, Embed_Dim=768

Pipeline status: READY FOR MULTI-HEAD ATTENTION
